# Task 1 — Character-Level GPT on TinyStories (Sarvesh)

From-scratch decoder-only Transformer for next-character prediction.

**What this notebook uses:** `nn.Linear`, `nn.Embedding`, `nn.LayerNorm`, `nn.Dropout`, and hand-written attention.

**What it does not use:** `nn.Transformer*`, `nn.MultiheadAttention`, `F.scaled_dot_product_attention`, or any HuggingFace / prebuilt attention module.

Implementation lives in the `charlm/` package (imported below). Hyperparameters are in `config.yaml`.

### Design choices (deliberately my own)
- Separate `W_q`, `W_k`, `W_v` projections (not a fused QKV matrix)
- Causal mask via `torch.triu` (block the future)
- Classic Transformer FFN with **ReLU** (Vaswani et al.)
- Token embeddings scaled by `sqrt(d_model)` + **weight tying**
- Story-shuffled train/val split (disjoint stories, seeded)
- Sliding windows with stride for training; warm-up + cosine LR to an absolute floor
- Nucleus (`top_p`) sampling in addition to temperature / top-k / greedy

In [ ]:
# Personal parameters — replace SEED with your course-assigned value if you have one.
SEED = 3141
print(f"SEED = {SEED}")

## 0) Setup

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

SRC_DIR = Path.cwd().resolve()
if SRC_DIR.name != "src":
    # Allow running with kernel cwd at repo root
    candidate = SRC_DIR / "src"
    if candidate.exists():
        SRC_DIR = candidate
        os.chdir(SRC_DIR)

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from charlm.config import load_config, resolve_paths
from charlm.data import decode_ids, make_loaders, prepare_datasets
from charlm.generate import run_generation_suite
from charlm.metrics import collect_full_metrics, write_metrics_report
from charlm.model import build_model, count_parameters
from charlm.plots import plot_loss_curves
from charlm.sanity import run_sanity_suite
from charlm.train import evaluate, train_model
from charlm.utils import amp_enabled, pick_device, set_seed

config_name = os.environ.get("TASK1_CONFIG", "config.yaml")
cfg = load_config(SRC_DIR / config_name)
cfg["seed"] = SEED  # notebook SEED wins over YAML

paths = resolve_paths(cfg, SRC_DIR)
device = pick_device(cfg.get("device", "auto"))
use_amp = amp_enabled(cfg, device)
set_seed(SEED)

print("config:", config_name)
print("device:", device, "| amp:", use_amp, "| torch:", torch.__version__)
print("run_name:", cfg["run_name"])
print("model:", cfg["model"])

## 1) Data preprocessing

1. Load TinyStories and split on `<|endoftext|>`
2. Seeded **story shuffle** → disjoint train / val story pools
3. Character vocab + integer encodings (`char_to_idx` / `idx_to_char`)
4. Fixed-length input–target windows for autoregressive LM (train uses stride)

In [ ]:
set_seed(SEED)
data = prepare_datasets(cfg, paths, SEED)
train_loader, val_loader = make_loaders(
    data["train_ds"],
    data["val_ds"],
    train_batch=cfg["train"]["batch_size"],
    eval_batch=cfg["eval"]["batch_size"],
    seed=SEED,
)

vocab_size = data["vocab_size"]
char_to_idx = data["char_to_idx"]
idx_to_char = data["idx_to_char"]

print(json.dumps(data["split_info"], indent=2))
print(f"vocab_size = {vocab_size}")
print(f"train windows = {len(data['train_ds'])} | val windows = {len(data['val_ds'])}")
print("chars sample:", repr("".join(data["chars"][:40])), "...")

xb, yb = next(iter(train_loader))
print("batch x/y:", tuple(xb.shape), tuple(yb.shape))
print("x[0][:80]:", repr(decode_ids(xb[0, :80].tolist(), idx_to_char)))
print("y[0][:80]:", repr(decode_ids(yb[0, :80].tolist(), idx_to_char)))

## 2) Model

```text
chars → token_embed * sqrt(d) + pos_embed → dropout
     → N × PreNormDecoderBlock(
            LN → MultiHeadCausalAttention(separate Q/K/V) → residual
            LN → FFN(ReLU) → residual
        )
     → LN → tied LM head → vocab logits
```

In [ ]:
set_seed(SEED)
model = build_model(cfg, vocab_size).to(device)
n_params = count_parameters(model)
print(f"parameters: {n_params:,}")
print(model)

## 3) Sanity checks (run before full training)

In [ ]:
sanity = run_sanity_suite(cfg, vocab_size, device)
assert sanity["causal_mask"], "Causal mask test failed — do not train"

## 4) Training

Cross-entropy next-character loss, AdamW, linear warm-up, cosine decay to `min_lr`.

In [ ]:
set_seed(SEED)
model = build_model(cfg, vocab_size)  # fresh weights for the real run
summary = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    paths=paths,
    device=device,
    seed=SEED,
)
print(json.dumps({k: v for k, v in summary.items() if k != "history"}, indent=2))

## 5) Loss curves

In [ ]:
curve_path = paths["output_dir"] / f"{cfg['run_name']}_loss_curves.png"
plot_loss_curves(summary["history"], curve_path, smooth_window=cfg["plots"]["smooth_window"])
print("saved:", curve_path)

from IPython.display import Image, display
display(Image(filename=str(curve_path)))

## 6) Load best checkpoint, metrics, generation

In [ ]:
ckpt_name = cfg["eval"].get("checkpoint", "best")
ckpt_path = paths["checkpoint_dir"] / f"{cfg['run_name']}_{ckpt_name}.pt"
blob = torch.load(ckpt_path, map_location=device)
model = build_model(cfg, vocab_size).to(device)
model.load_state_dict(blob["model"])
model.eval()
print("loaded:", ckpt_path, "| best_epoch:", blob.get("best_epoch"))

generated, speeds, gen_json = run_generation_suite(
    model=model,
    cfg=cfg,
    char_to_idx=char_to_idx,
    idx_to_char=idx_to_char,
    device=device,
    seed=SEED,
    output_dir=paths["output_dir"],
)

rows = collect_full_metrics(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    cfg=cfg,
    device=device,
    use_amp=use_amp,
    summary=summary,
    generated=generated,
    gen_tokens_per_sec=speeds,
)
metrics_path = paths["metrics_csv"]
write_metrics_report(metrics_path, rows)
print("metrics ->", metrics_path)

for setting, texts in generated.items():
    print("\n========", setting, "========")
    print(texts[0][:500])
    print("...")

## 7) Failure analysis notes

After generation finishes, open samples under `outputs/<run>_samples/` and fill in `failure_analysis.md` with three failure cases (repetition, broken grammar, coherence loss, hallucination, etc.).

Also update `results.md` with the numbers from `metrics_report.csv`.